In [2]:
from bs4 import BeautifulSoup, Tag
from datetime import datetime
from tqdm import tqdm
import pandas as pd
import requests
from utils.helper_functions import get_path


In [16]:
def get_screener_url(company_or_ticker: str) -> str:
    # Using screener api to get the exact url to hit for the required company
    results = requests.get(
        f"https://www.screener.in/api/company/search/?q={company_or_ticker}"
    )
    valid_results = [
        res for res in results.json()
        if not any(term in res["name"].lower() for term in ["partly", "paid", "rights", "warrant", "re-"])
    ]
    
    if not valid_results:
        print(f"No valid company found for ticker: {company_or_ticker}")
        return ''

    primary_company = valid_results[0]
    primary_company_path = primary_company['url']
    complete_screener_url = f"https://www.screener.in{primary_company_path}"
    
    return complete_screener_url

def latest_qtr_result(screener_url: str) -> pd.DataFrame:
    response = requests.get(screener_url, headers={"User-Agent": "Mozilla/5.0"})
    soup = BeautifulSoup(response.text, "html.parser")

    # 1. Target the Quarterly Results section directly
    quarters_section = soup.find("section", id="quarters")
    if not quarters_section:
        raise ValueError(f"Could not find quarterly results section on {screener_url}")

    table = quarters_section.find("table")
    if not table:
        raise ValueError(f"Could not find quarterly table in {screener_url}")

    # 2. Extract headers
    headers = [th.get_text(strip=True) for th in table.find_all("th")]

    # 3. Extract rows
    rows = []
    for tr in table.find_all("tr"):
        cells = tr.find_all("td")
        if not cells:
            continue
        row = [td.get_text(strip=True) for td in cells]
        rows.append(row)

    # 4. Create DataFrame
    quarterly_df = pd.DataFrame(rows, columns=headers)
    return quarterly_df


In [17]:
out = get_screener_url('pvv')
out

'https://www.screener.in/company/536659/'

In [18]:
output = latest_qtr_result(out)
output

,,Jun 2023,Sep 2023,Dec 2023,Mar 2024,Jun 2024,Sep 2024,Dec 2024,Mar 2025,Jun 2025,Sep 2025,Dec 2025,Mar 2026,Jun 2026
0,Sales+,33.88,23.36,18.54,12.15,21.39,3.53,8.25,6.68,7.83,10.93,16.24,21.35,8.50
1,Expenses+,33.91,22.98,18.72,9.83,15.72,3.51,8.11,7.40,6.96,8.44,11.29,21.23,7.59
2,Operating Profit,-0.03,0.38,-0.18,2.32,5.67,0.02,0.14,-0.72,0.87,2.49,4.95,0.12,0.91
3,OPM %,-0.09%,1.63%,-0.97%,19.09%,26.51%,0.57%,1.70%,-10.78%,11.11%,22.78%,30.48%,0.56%,10.71%
4,Other Income+,0.04,0.00,1.27,-0.17,0.00,0.00,0.00,1.34,0.00,0.00,0.00,0.00,0.00
5,Interest,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
6,Depreciation,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
7,Profit before tax,0.01,0.38,1.09,2.15,5.67,0.02,0.14,0.62,0.87,2.49,4.95,0.12,0.91
8,Tax %,0.00%,15.79%,15.60%,14.88%,14.99%,0.00%,14.29%,14.52%,14.94%,14.86%,14.95%,16.67%,15.38%
9,Net Profit+,0.01,0.33,0.92,1.83,4.82,0.02,0.12,0.53,0.74,2.11,4.21,0.11,0.77
